# NOTEBOOK 04: ADVANCED GEOSPATIAL & SPATIAL ANALYSIS (UPGRADED)

## Air Quality Monitoring - European Cities

## Focus: City Boundaries, Interactive Maps, Advanced Spatial Statistics

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import folium
from folium.plugins import HeatMap, MarkerCluster, MiniMap, Fullscreen
import geopandas as gpd
from shapely.geometry import Point, MultiPoint, box
from shapely.ops import unary_union
import json
from scipy.spatial import ConvexHull, Voronoi
from scipy.spatial.distance import cdist
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KernelDensity
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# SETUP & PATHS

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
import os
BASE_PATH = '/content/drive/MyDrive/Project Mandiri/Air Quality Monitoring in European Cities'
CLEANED_DATA_PATH = f'{BASE_PATH}/data/cleaned'
RESULTS_GEOSPATIAL = f'{BASE_PATH}/results/geospatial'

os.makedirs(RESULTS_GEOSPATIAL, exist_ok=True)

print("✅ Setup complete")

✅ Setup complete


# LOAD & PREPARE DATA

## LOADING & PREPARING ADVANCED GEOSPATIAL DATA

In [4]:
df_all = pd.read_csv(f'{CLEANED_DATA_PATH}/all_cities_cleaned_with_anomalies.csv')
df_all['date'] = pd.to_datetime(df_all['date'])

In [23]:
print(
    df_all[df_all['city'] == 'Zaragoza'][['station_name', 'pm25']]
    .groupby('station_name')
    .agg(
        jumlah_data=('pm25', 'size'),
        jumlah_nan=('pm25', lambda x: x.isna().sum()),
        mean_pm25=('pm25', 'mean')
    )
)

                         jumlah_data  jumlah_nan  mean_pm25
station_name                                               
Actur                          24767       24767        NaN
Avenida Soria                  24767       24767        NaN
Centro                         24767       24767        NaN
El Picarral                    15456       15456        NaN
Jaime Ferran                   24767       24767        NaN
Las Fuentes                    24767       24767        NaN
Renovales                      24744       24744        NaN
Roger De Flor                  24767       24767        NaN
Zaragoza CAMS station 0        24767       24767        NaN
Zaragoza CAMS station 1        24767       24767        NaN


In [5]:
# Calculate location statistics
location_stats = df_all.groupby(['latitude', 'longitude', 'city', 'station_name']).agg({
    'pm25': ['mean', 'std', 'min', 'max'],
    'pm10': ['mean', 'std'],
    'no2': ['mean', 'std'],
    'o3': ['mean', 'std'],
    'temperature': 'mean',
    'relative_humidity': 'mean',
    'wind_u': 'mean',
    'wind_v': 'mean'
}).reset_index()

location_stats.columns = ['_'.join(col).strip('_') for col in location_stats.columns.values]
location_stats.rename(columns={
    'latitude': 'lat',
    'longitude': 'lon',
    'city': 'city',
    'station_name': 'station'
}, inplace=True)

# Create GeoDataFrame
geometry = [Point(xy) for xy in zip(location_stats['lon'], location_stats['lat'])]
gdf_stations = gpd.GeoDataFrame(location_stats, geometry=geometry, crs='EPSG:4326')

print(f"✅ Data loaded: {len(gdf_stations)} stations")
print(f"   Cities: {gdf_stations['city'].unique()}")

# Color palette
COLORS_CITIES = {
    'Ancona': {'light': '#3498DB', 'dark': '#2980B9', 'fill': '#AED6F1'},
    'Athens': {'light': '#E74C3C', 'dark': '#C0392B', 'fill': '#F5B7B1'},
    'Zaragoza': {'light': '#2ECC71', 'dark': '#27AE60', 'fill': '#ABEBC6'}
}

BG_COLOR = 'rgba(245, 247, 250, 1)'
ACCENT_COLOR = '#1f77b4'
TEXT_COLOR = '#2C3E50'

✅ Data loaded: 96 stations
   Cities: ['Athens' 'Zaragoza' 'Ancona']


# SECTION 1: CREATE CITY BOUNDARIES & CONVEX HULLS

In [6]:
city_boundaries = {}
city_hulls = {}

for city in ['Ancona', 'Athens', 'Zaragoza']:
    print(f"\n🏙️  Processing {city}...")

    gdf_city = gdf_stations[gdf_stations['city'] == city]

    # Create convex hull (boundary yang mengelilingi semua stations)
    points = np.array([[p.x, p.y] for p in gdf_city.geometry])

    if len(points) >= 3:
        hull = ConvexHull(points)
        hull_coords = points[hull.vertices]
        hull_polygon = gpd.GeoSeries([Point(coord).buffer(0.01) for coord in hull_coords]).unary_union.convex_hull

        # Better: Use shapely to create polygon directly from hull
        from shapely.geometry import Polygon
        hull_polygon = Polygon(hull_coords)

        city_boundaries[city] = hull_polygon
        city_hulls[city] = hull_polygon

        # Calculate city statistics
        center_lat = gdf_city['lat'].mean()
        center_lon = gdf_city['lon'].mean()
        area = hull_polygon.area

        print(f"   ✓ Convex hull created")
        print(f"   ✓ Center: ({center_lat:.4f}, {center_lon:.4f})")
        print(f"   ✓ Area: {area:.4f} sq degrees")
        print(f"   ✓ Stations: {len(gdf_city)}")


🏙️  Processing Ancona...
   ✓ Convex hull created
   ✓ Center: (43.5678, 13.3634)
   ✓ Area: 0.0800 sq degrees
   ✓ Stations: 22

🏙️  Processing Athens...
   ✓ Convex hull created
   ✓ Center: (37.9983, 23.7427)
   ✓ Area: 0.1290 sq degrees
   ✓ Stations: 64

🏙️  Processing Zaragoza...
   ✓ Convex hull created
   ✓ Center: (41.6554, -0.8885)
   ✓ Area: 0.0024 sq degrees
   ✓ Stations: 10


# SECTION 2: INTERACTIVE CHOROPLETH MAP - CITY BOUNDARIES WITH POLLUTION

In [7]:
# Create multi-layer interactive map dengan city boundaries
m = folium.Map(
    location=[41.0, 12.0],  # Center antara 3 cities
    zoom_start=5,
    tiles='OpenStreetMap',
    attr='© OpenStreetMap contributors'
)

# Layer control untuk toggle cities
feature_groups = {}

In [20]:
for city in ['Ancona', 'Athens', 'Zaragoza']:

    gdf_city = gdf_stations[gdf_stations['city'] == city]

    # ==========================
    # Data untuk HeatMap & Marker
    # ==========================
    gdf_heat = gdf_city.dropna(subset=['lat', 'lon', 'pm25_mean']).copy()

    gdf_marker = gdf_city.dropna(
        subset=[
            'lat',
            'lon',
            'pm25_mean',
            'pm10_mean',
            'no2_mean',
            'o3_mean',
            'temperature_mean',
            'relative_humidity_mean',
            'wind_u_mean',
            'wind_v_mean'
        ]
    ).copy()

    # ==========================
    # Feature Group
    # ==========================
    fg = folium.FeatureGroup(
        name=f'<b style="color:{COLORS_CITIES[city]["dark"]}">{city}</b>',
        show=True
    )

    # ==========================
    # Boundary
    # ==========================
    boundary = city_boundaries[city]
    boundary_coords = list(boundary.exterior.coords)

    folium.Polygon(
        locations=[[p[1], p[0]] for p in boundary_coords],
        color=COLORS_CITIES[city]['dark'],
        fill=True,
        fillColor=COLORS_CITIES[city]['fill'],
        fillOpacity=0.3,
        weight=3,
        popup=f"<b>{city}</b> City Boundary",
        tooltip=city
    ).add_to(fg)

    # ==========================
    # HeatMap
    # ==========================
    heat_data = [
        [row['lat'], row['lon'], row['pm25_mean']]
        for _, row in gdf_heat.iterrows()
    ]

    if len(heat_data) > 0:

        HeatMap(
            heat_data,
            name=f"{city} PM2.5 Heat",
            radius=20,
            blur=15,
            max_zoom=1,
            max=100,
            min_opacity=0.2,
            gradient={
                0.2: 'blue',
                0.4: 'lime',
                0.6: 'yellow',
                0.8: 'orange',
                1.0: 'red'
            }
        ).add_to(fg)

    else:

        print(f"⚠ Tidak ada data HeatMap untuk {city}")

    # ==========================
    # Marker Cluster
    # ==========================
    marker_cluster = MarkerCluster(
        name=f"{city} Stations"
    ).add_to(fg)

    print(city)
    print(len(gdf_heat))
    print(len(gdf_marker))

    for _, row in gdf_marker.iterrows():

        pm25 = row['pm25_mean']

        if pm25 <= 15:
            color = "green"
            aqi = "Good"
        elif pm25 <= 35:
            color = "yellow"
            aqi = "Moderate"
        elif pm25 <= 75:
            color = "orange"
            aqi = "Unhealthy for Sensitive"
        else:
            color = "red"
            aqi = "Very Unhealthy"

        wind_u = row['wind_u_mean']
        wind_v = row['wind_v_mean']

        wind_speed = np.sqrt(wind_u**2 + wind_v**2)

        popup_html = f"""
        <div style="font-family:Arial;width:280px;">
            <b style="font-size:14px;color:{COLORS_CITIES[city]['dark']}">
                {row['station']}
            </b>
            <hr>

            <table style="font-size:12px;width:100%;">
                <tr><td><b>PM2.5</b></td><td>{row['pm25_mean']:.1f}</td></tr>
                <tr><td><b>PM10</b></td><td>{row['pm10_mean']:.1f}</td></tr>
                <tr><td><b>NO₂</b></td><td>{row['no2_mean']:.1f}</td></tr>
                <tr><td><b>O₃</b></td><td>{row['o3_mean']:.1f}</td></tr>
                <tr><td><b>Temperature</b></td><td>{row['temperature_mean']:.1f} °C</td></tr>
                <tr><td><b>Humidity</b></td><td>{row['relative_humidity_mean']:.1f}%</td></tr>
                <tr><td><b>Wind Speed</b></td><td>{wind_speed:.2f} m/s</td></tr>
                <tr><td><b>AQI</b></td><td>{aqi}</td></tr>
            </table>

            <hr>

            <small>
                {row['lat']:.4f},
                {row['lon']:.4f}
            </small>

        </div>
        """

        folium.CircleMarker(
            location=[row['lat'], row['lon']],
            radius=8,
            popup=folium.Popup(popup_html, max_width=300),
            color=COLORS_CITIES[city]['dark'],
            fill=True,
            fillColor=color,
            fillOpacity=0.8,
            weight=2,
            tooltip=f"{row['station']}<br>PM2.5: {row['pm25_mean']:.1f}"
        ).add_to(marker_cluster)

    feature_groups[city] = fg
    fg.add_to(m)

# Add layer control untuk toggle cities
folium.LayerControl(position='topright', collapsed=False).add_to(m)

# Add fullscreen button
Fullscreen(position='topleft').add_to(m)

# Add mini map
minimap = MiniMap(toggle_display=True)
m.add_child(minimap)

# Add legend
legend_html = '''
<div style="position: fixed;
     bottom: 80px; right: 50px; width: 220px; height: auto;
     background-color: white; border:3px solid #2C3E50; z-index:9999;
     font-size:13px; padding: 15px; border-radius: 8px;
     box-shadow: 0 2px 6px rgba(0,0,0,0.3);">
     <h4 style="margin: 0 0 10px 0; color: #1f77b4;">Air Quality Legend</h4>
     <hr style="margin: 5px 0; border: none; border-top: 1px solid #ccc;">
     <div style="margin: 5px 0;"><i class="fa fa-circle" style="color:green"></i> Good (≤15 µg/m³)</div>
     <div style="margin: 5px 0;"><i class="fa fa-circle" style="color:yellow"></i> Moderate (15-35)</div>
     <div style="margin: 5px 0;"><i class="fa fa-circle" style="color:orange"></i> Unhealthy (35-75)</div>
     <div style="margin: 5px 0;"><i class="fa fa-circle" style="color:red"></i> Very Unhealthy (>75)</div>
     <hr style="margin: 8px 0; border: none; border-top: 1px solid #ccc;">
     <small><b>Click cities in legend to toggle visibility</b></small>
</div>
'''

m.get_root().html.add_child(folium.Element(legend_html))

m

Ancona
22
22
Athens
64
64
⚠ Tidak ada data HeatMap untuk Zaragoza
Zaragoza
0
0


In [21]:
# Save map
m.save(f'{RESULTS_GEOSPATIAL}/01_interactive_city_boundaries_map.html')
print("\n✅ Saved: 01_interactive_city_boundaries_map.html")


✅ Saved: 01_interactive_city_boundaries_map.html


# SECTION 3: INDIVIDUAL CITY DETAIL MAPS - ZOOMED WITH ANALYSIS

In [30]:
for city in ['Ancona', 'Athens', 'Zaragoza']:
    print(f"\n🔍 Creating detailed map for {city}...")

    gdf_city = gdf_stations[gdf_stations['city'] == city]

    # Hapus data yang tidak memiliki PM2.5
    gdf_heat = gdf_city.dropna(subset=['lat', 'lon', 'pm25_mean']).copy()

    center_lat = gdf_city['lat'].mean()
    center_lon = gdf_city['lon'].mean()

    # Create detailed map
    m_detail = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=11,
        tiles='OpenStreetMap'
    )

    # Add city boundary
    boundary = city_boundaries[city]
    boundary_coords = list(boundary.exterior.coords)

    folium.Polygon(
        locations=[[p[1], p[0]] for p in boundary_coords],
        color=COLORS_CITIES[city]['dark'],
        fill=True,
        fillColor=COLORS_CITIES[city]['fill'],
        fillOpacity=0.25,
        weight=4,
        popup=f'<b>{city}</b> Monitoring Area',
        tooltip=f'{city} Boundary'
    ).add_to(m_detail)

    # Create radius analysis
    radii = [0.01, 0.02, 0.03, 0.05]
    colors_radius = ['#3498DB', '#2ECC71', '#E74C3C', '#F39C12']

    for radius, color in zip(radii, colors_radius):
        folium.Circle(
            location=[center_lat, center_lon],
            radius=radius * 111000,
            color=color,
            fill=False,
            weight=1,
            opacity=0.5,
            popup=f'Radius: {radius*111:.0f} km',
            tooltip=f'{radius*111:.0f} km radius'
        ).add_to(m_detail)

    # =========================
    # HEATMAP
    # =========================

    heat_data = [
        [row['lat'], row['lon'], row['pm25_mean']]
        for _, row in gdf_heat.iterrows()
    ]

    if len(heat_data) > 0:

        HeatMap(
            heat_data,
            radius=25,
            blur=20,
            max_zoom=1,
            max=100,
            min_opacity=0.3,
            gradient={
                0.2: 'blue',
                0.4: 'cyan',
                0.6: 'lime',
                0.8: 'yellow',
                0.9: 'orange',
                1.0: 'red'
            }
        ).add_to(m_detail)

    else:
        print(f"⚠ {city}: HeatMap dilewati karena seluruh PM2.5 kosong.")

    # =========================
    # MARKER
    # =========================

    gdf_marker = gdf_city.dropna(
        subset=[
            'lat',
            'lon',
            'pm10_mean',
            'no2_mean',
            'o3_mean',
            'temperature_mean',
            'relative_humidity_mean',
            'wind_u_mean',
            'wind_v_mean'
        ]
    ).copy()

    # Add detailed station markers
    for _, row in gdf_marker.iterrows():

        if pd.isna(row['pm25_mean']):
            color = "gray"
            aqi = "No PM2.5 Data"
            pm25_text = "N/A"

        else:
            pm25 = row['pm25_mean']
            pm25_text = f"{pm25:.1f}"

            if pm25 <= 15:
                color = "green"
                aqi = "Good"

            elif pm25 <= 35:
                color = "yellow"
                aqi = "Moderate"

            elif pm25 <= 75:
                color = "orange"
                aqi = "Unhealthy for Sensitive"

            else:
                color = "red"
                aqi = "Very Unhealthy"

        # Wind data
        wind_u = row['wind_u_mean']
        wind_v = row['wind_v_mean']

        wind_speed = np.sqrt(wind_u**2 + wind_v**2)
        wind_dir = np.degrees(np.arctan2(wind_v, wind_u))

        if wind_dir < 0:
            wind_dir += 360

        if wind_dir < 45 or wind_dir >= 315:
            cardinal = 'N'
        elif wind_dir < 135:
            cardinal = 'E'
        elif wind_dir < 225:
            cardinal = 'S'
        else:
            cardinal = 'W'

        popup_html = f"""
        <div style="font-family: Arial; width: 300px; font-size: 12px;">
            <h4 style="margin: 5px 0; color: {COLORS_CITIES[city]['dark']}">{row['station']}</h4>
            <hr style="margin: 5px 0;">

            <b>Pollutants:</b><br>
            • PM2.5: {pm25_text} µg/m³ (std: {row['pm25_std']:.1f})<br>
            • PM10: {row['pm10_mean']:.1f} µg/m³ (std: {row['pm10_std']:.1f})<br>
            • NO₂: {row['no2_mean']:.1f} µg/m³ (std: {row['no2_std']:.1f})<br>
            • O₃: {row['o3_mean']:.1f} µg/m³ (std: {row['o3_std']:.1f})<br>

            <b>Meteorology:</b><br>
            • Temperature: {row['temperature_mean']:.1f}°C<br>
            • Humidity: {row['relative_humidity_mean']:.1f}%<br>
            • Wind: {wind_speed:.2f} m/s from {cardinal} ({wind_dir:.0f}°)<br>

            <b>AQI Status:</b>
            <span style="color:{color}; font-weight:bold;">{aqi}</span><br>

            <hr style="margin:5px 0;">
            <small>
            Lat: {row['lat']:.6f}<br>
            Lon: {row['lon']:.6f}
            </small>
        </div>
        """

        folium.CircleMarker(
            location=[row['lat'], row['lon']],
            radius=10,
            popup=folium.Popup(popup_html, max_width=350),
            color=COLORS_CITIES[city]['dark'],
            fill=True,
            fillColor=color,
            fillOpacity=0.85,
            weight=2.5,
            opacity=0.95,
            tooltip=f"{row['station']}<br>PM2.5: {pm25_text} µg/m³<br>Wind: {cardinal} {wind_speed:.1f} m/s"
        ).add_to(m_detail)

        # Draw wind arrow
        if wind_speed > 0.5:

            arrow_length = min(0.01, wind_speed * 0.005)

            arrow_lat = row['lat'] + arrow_length * np.cos(np.radians(wind_dir))
            arrow_lon = row['lon'] + arrow_length * np.sin(np.radians(wind_dir))

            folium.PolyLine(
                [[row['lat'], row['lon']], [arrow_lat, arrow_lon]],
                color=COLORS_CITIES[city]['dark'],
                weight=2,
                opacity=0.6,
                popup='Wind direction'
            ).add_to(m_detail)

    # Add fullscreen & minimap
    Fullscreen(position='topleft').add_to(m_detail)
    MiniMap(toggle_display=True).add_to(m_detail)

    # Add layer control
    folium.LayerControl().add_to(m_detail)

    # Save detailed map
    m_detail.save(f'{RESULTS_GEOSPATIAL}/02_detailed_map_{city.lower()}.html')

    print(f"   ✅ Saved: 02_detailed_map_{city.lower()}.html")


🔍 Creating detailed map for Ancona...
   ✅ Saved: 02_detailed_map_ancona.html

🔍 Creating detailed map for Athens...
   ✅ Saved: 02_detailed_map_athens.html

🔍 Creating detailed map for Zaragoza...
⚠ Zaragoza: HeatMap dilewati karena seluruh PM2.5 kosong.
   ✅ Saved: 02_detailed_map_zaragoza.html


# SECTION 4: SPATIAL DENSITY ANALYSIS (KERNEL DENSITY ESTIMATION)

In [32]:
kde_results = {}

for city in ['Ancona', 'Athens', 'Zaragoza']:
    print(f"\n📊 KDE Analysis for {city}...")

    # ==================================================
    # Ambil data kota dan hapus data yang memiliki NaN
    # ==================================================
    gdf_city = gdf_stations[
        gdf_stations['city'] == city
    ].dropna(subset=['lat', 'lon', 'pm25_mean']).copy()

    # Jika data terlalu sedikit, lewati kota tersebut
    if len(gdf_city) < 2:
        print(f"⚠ {city}: Tidak cukup data untuk KDE.")
        continue

    # ==================================================
    # Create grid
    # ==================================================
    lat_range = np.linspace(
        gdf_city['lat'].min() - 0.05,
        gdf_city['lat'].max() + 0.05,
        60
    )

    lon_range = np.linspace(
        gdf_city['lon'].min() - 0.05,
        gdf_city['lon'].max() + 0.05,
        60
    )

    lat_grid, lon_grid = np.meshgrid(lat_range, lon_range)

    grid_points = np.column_stack(
        (lon_grid.ravel(), lat_grid.ravel())
    )

    # ==================================================
    # KDE
    # ==================================================
    points = gdf_city[['lon', 'lat']].values
    values = gdf_city['pm25_mean'].values.astype(float)

    kde = KernelDensity(
        bandwidth=0.02,
        kernel='gaussian'
    )

    kde.fit(points, sample_weight=values)

    density = kde.score_samples(grid_points)
    density = np.exp(density).reshape(lat_grid.shape)

    kde_results[city] = {
        'density': density,
        'lat_grid': lat_grid,
        'lon_grid': lon_grid,
        'points': points,
        'values': values
    }

    print(f"   ✓ KDE completed for {city}")

# ==================================================
# Visualisasi KDE
# ==================================================

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=['Ancona', 'Athens', 'Zaragoza'],
    specs=[[{'type': 'contour'},
            {'type': 'contour'},
            {'type': 'contour'}]]
)

for col_idx, city in enumerate(['Ancona', 'Athens', 'Zaragoza'], start=1):

    if city not in kde_results:
        continue

    result = kde_results[city]

    fig.add_trace(
        go.Contour(
            z=result['density'],
            x=result['lon_grid'][0],
            y=result['lat_grid'][:, 0],
            colorscale='Viridis',
            name='PM2.5 Density',
            showscale=(col_idx == 3),
            colorbar=dict(
                title='Density',
                x=1.02
            ) if col_idx == 3 else None,
            hovertemplate='Lat: %{y:.4f}<br>Lon: %{x:.4f}<br>Density: %{z:.4f}<extra></extra>'
        ),
        row=1,
        col=col_idx
    )

    fig.add_trace(
        go.Scatter(
            x=result['points'][:, 0],
            y=result['points'][:, 1],
            mode='markers',
            marker=dict(
                size=7,
                color=result['values'],
                colorscale='RdYlGn_r',
                showscale=False,
                line=dict(
                    color='white',
                    width=1
                )
            ),
            text=[f"PM2.5: {v:.1f}" for v in result['values']],
            name='Stations',
            hovertemplate='<b>Station</b><br>%{text}<extra></extra>',
            showlegend=(col_idx == 1)
        ),
        row=1,
        col=col_idx
    )

    fig.update_xaxes(
        title_text='Longitude',
        row=1,
        col=col_idx
    )

    fig.update_yaxes(
        title_text='Latitude',
        row=1,
        col=col_idx
    )

fig.update_layout(
    title_text='Kernel Density Estimation - PM2.5 Spatial Distribution',
    title_font_size=24,
    title_font_color=ACCENT_COLOR,
    height=550,
    paper_bgcolor='white',
    font=dict(
        size=11,
        color=TEXT_COLOR
    ),
    margin=dict(
        l=80,
        r=120,
        t=100,
        b=80
    )
)

fig.write_html(
    f'{RESULTS_GEOSPATIAL}/03_kernel_density_estimation.html'
)

print("\n✅ Saved: 03_kernel_density_estimation.html")

fig.show()


📊 KDE Analysis for Ancona...
   ✓ KDE completed for Ancona

📊 KDE Analysis for Athens...
   ✓ KDE completed for Athens

📊 KDE Analysis for Zaragoza...
⚠ Zaragoza: Tidak cukup data untuk KDE.

✅ Saved: 03_kernel_density_estimation.html


# SECTION 5: VORONOI DIAGRAM - SERVICE AREAS

In [33]:
print("SECTION 5: VORONOI DIAGRAM - MONITORING STATION SERVICE AREAS")

SECTION 5: VORONOI DIAGRAM - MONITORING STATION SERVICE AREAS


In [34]:
for city in ['Ancona', 'Athens', 'Zaragoza']:
    print(f"\n🔷 Creating Voronoi for {city}...")

    gdf_city = gdf_stations[gdf_stations['city'] == city]

    # Create Voronoi diagram
    points = gdf_city[['lon', 'lat']].values

    if len(points) >= 3:
        vor = Voronoi(points)

        # Create interactive map
        m_vor = folium.Map(
            location=[gdf_city['lat'].mean(), gdf_city['lon'].mean()],
            zoom_start=10,
            tiles='OpenStreetMap'
        )

        # Draw Voronoi regions
        for i, region_idx in enumerate(vor.point_region):
            region = vor.regions[region_idx]

            if -1 not in region and len(region) > 0:
                vertices = vor.vertices[region]
                vertices = np.vstack([vertices, vertices[0]])  # Close polygon

                folium.PolyLine(
                    locations=[[v[1], v[0]] for v in vertices],
                    color=COLORS_CITIES[city]['light'],
                    weight=2,
                    opacity=0.7,
                    popup=f'Service Area {i}'
                ).add_to(m_vor)

        # Add station points
        for idx, row in gdf_city.iterrows():
            pm25 = row['pm25_mean']
            color = 'green' if pm25 <= 15 else ('yellow' if pm25 <= 35 else ('orange' if pm25 <= 75 else 'red'))

            folium.CircleMarker(
                location=[row['lat'], row['lon']],
                radius=10,
                popup=f"{row['station']}<br>PM2.5: {row['pm25_mean']:.1f}",
                color=COLORS_CITIES[city]['dark'],
                fill=True,
                fillColor=color,
                fillOpacity=0.85,
                weight=2.5
            ).add_to(m_vor)

        # Add legend
        legend_html = f'''
        <div style="position: fixed;
             bottom: 50px; right: 50px; width: 200px; height: auto;
             background-color: white; border:2px solid {COLORS_CITIES[city]['dark']}; z-index:9999;
             font-size:12px; padding: 10px; border-radius: 5px;">
             <b style="color: {COLORS_CITIES[city]['dark']}">{city} Voronoi Diagram</b><br>
             <hr style="margin: 5px 0;">
             Lines = Service boundaries<br>
             Each station monitors<br>
             its Voronoi region
        </div>
        '''

        m_vor.get_root().html.add_child(folium.Element(legend_html))

        Fullscreen(position='topleft').add_to(m_vor)
        m_vor.save(f'{RESULTS_GEOSPATIAL}/04_voronoi_diagram_{city.lower()}.html')
        print(f"   ✅ Saved: 04_voronoi_diagram_{city.lower()}.html")


🔷 Creating Voronoi for Ancona...
   ✅ Saved: 04_voronoi_diagram_ancona.html

🔷 Creating Voronoi for Athens...
   ✅ Saved: 04_voronoi_diagram_athens.html

🔷 Creating Voronoi for Zaragoza...
   ✅ Saved: 04_voronoi_diagram_zaragoza.html


# SECTION 6: DISTANCE MATRIX & PROXIMITY ANALYSIS

In [35]:
# For each city, calculate station-to-station distances
proximity_data = []

for city in ['Ancona', 'Athens', 'Zaragoza']:
    gdf_city = gdf_stations[gdf_stations['city'] == city]

    points = gdf_city[['lon', 'lat']].values

    # Convert lat/lon to kilometers (approximate)
    # 1 degree latitude ≈ 111 km, 1 degree longitude ≈ 111 * cos(lat) km

    distances = cdist(points, points, metric='euclidean') * 111  # Approximate km

    # Find nearest neighbors
    for i, (idx, row) in enumerate(gdf_city.iterrows()):
        nearest_idx = np.argsort(distances[i])[1]  # Skip self (index 0)
        nearest_dist = distances[i][nearest_idx]
        nearest_station = gdf_city.iloc[nearest_idx]['station']

        proximity_data.append({
            'City': city,
            'Station': row['station'],
            'Nearest Station': nearest_station,
            'Distance (km)': nearest_dist,
            'Pollution Diff (PM2.5)': abs(row['pm25_mean'] - gdf_city.iloc[nearest_idx]['pm25_mean'])
        })

proximity_df = pd.DataFrame(proximity_data)

print("\n📊 PROXIMITY ANALYSIS - NEAREST NEIGHBORS:")
print(proximity_df.to_string(index=False))


📊 PROXIMITY ANALYSIS - NEAREST NEIGHBORS:
    City                                 Station                         Nearest Station  Distance (km)  Pollution Diff (PM2.5)
  Ancona                  Ancona CAMS Station 10                   Ancona CAMS Station 5      11.100000                0.816957
  Ancona                  Ancona CAMS Station 11                                    Jesi       8.663881                2.058895
  Ancona                  Ancona CAMS Station 12                  Ancona CAMS Station 13      11.099936                0.069502
  Ancona                  Ancona CAMS Station 13                  Ancona CAMS Station 12      11.099936                0.069502
  Ancona                  Ancona CAMS Station 14                   Ancona CAMS Station 9      11.100000                0.577750
  Ancona                                    Jesi                   Ancona CAMS Station 6       2.471758                0.899380
  Ancona                   Ancona CAMS Station 5             

In [ ]:
# Save proximity data
proximity_df.to_csv(f'{RESULTS_GEOSPATIAL}/proximity_analysis.csv', index=False)
print(f"\n✅ Saved: proximity_analysis.csv")

In [36]:
# Visualize proximity
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Distance to Nearest Station', 'Pollution Difference from Nearest'],
    specs=[[{'type': 'box'}, {'type': 'box'}]]
)

for city, color in COLORS_CITIES.items():
    city_data = proximity_df[proximity_df['City'] == city]

    fig.add_trace(
        go.Box(
            y=city_data['Distance (km)'],
            name=city,
            marker=dict(color=color['light']),
            showlegend=True
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Box(
            y=city_data['Pollution Diff (PM2.5)'],
            name=city,
            marker=dict(color=color['light']),
            showlegend=False
        ),
        row=1, col=2
    )

fig.update_yaxes(title_text='Distance (km)', row=1, col=1)
fig.update_yaxes(title_text='PM2.5 Difference (µg/m³)', row=1, col=2)

fig.update_layout(
    title_text='Proximity Analysis - Distance & Pollution Difference to Nearest Station',
    title_font_size=22,
    title_font_color=ACCENT_COLOR,
    height=550,
    plot_bgcolor=BG_COLOR,
    paper_bgcolor='white',
    hovermode='closest',
    font=dict(size=12, color=TEXT_COLOR),
    margin=dict(l=80, r=50, t=100, b=80)
)

fig.write_html(f'{RESULTS_GEOSPATIAL}/05_proximity_analysis.html')
print("✅ Saved: 05_proximity_analysis.html")
fig.show()

✅ Saved: 05_proximity_analysis.html


# SECTION 7: ADVANCED CLUSTERING STATISTICS

In [37]:
clustering_stats = []

for city in ['Ancona', 'Athens', 'Zaragoza']:
    gdf_city = gdf_stations[gdf_stations['city'] == city]

    points = gdf_city[['lon', 'lat']].values
    values = gdf_city['pm25_mean'].values

    # Calculate clustering metrics
    points_scaled = points * np.array([111 * np.cos(np.radians(points[:, 1].mean())), 111])

    # Average distance to nearest neighbor
    distances = cdist(points_scaled, points_scaled)
    np.fill_diagonal(distances, np.inf)
    avg_nn_distance = np.mean(np.min(distances, axis=1))

    # Spatial dispersion (variance in coordinates)
    dispersion = np.std(points)

    # Pollution clustering (coefficient of variation in values)
    pollution_cv = np.std(values) / np.mean(values)

    clustering_stats.append({
        'City': city,
        'Num Stations': len(gdf_city),
        'Avg Dist to Nearest (km)': avg_nn_distance,
        'Spatial Dispersion': dispersion,
        'Pollution CV': pollution_cv,
        'Mean PM2.5': np.mean(values),
        'PM2.5 Std': np.std(values)
    })

clustering_stats_df = pd.DataFrame(clustering_stats)

print("\n📊 CLUSTERING STATISTICS:")
print(clustering_stats_df.to_string(index=False))


📊 CLUSTERING STATISTICS:
    City  Num Stations  Avg Dist to Nearest (km)  Spatial Dispersion  Pollution CV  Mean PM2.5  PM2.5 Std
  Ancona            22                  5.184931           15.102546      0.111443   13.013333   1.450246
  Athens            64                  2.747327            7.128322      0.251299   15.074738   3.788265
Zaragoza            10                  1.403835           21.271961           NaN         NaN        NaN


In [ ]:
# Save stats
clustering_stats_df.to_csv(f'{RESULTS_GEOSPATIAL}/clustering_statistics.csv', index=False)
print(f"\n✅ Saved: clustering_statistics.csv")

# SECTION 8: COMPREHENSIVE GEOSPATIAL ANALYSIS REPORT

In [38]:
spatial_report = f"""
# ADVANCED GEOSPATIAL & SPATIAL ANALYSIS REPORT
Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}

## GEOGRAPHIC OVERVIEW

### City Boundaries & Coverage
- Ancona: {len(gdf_stations[gdf_stations['city'] == 'Ancona'])} monitoring stations
- Athens: {len(gdf_stations[gdf_stations['city'] == 'Athens'])} monitoring stations
- Zaragoza: {len(gdf_stations[gdf_stations['city'] == 'Zaragoza'])} monitoring stations

Total: {len(gdf_stations)} stations across Europe

### Spatial Extent (Convex Hulls)
Each city has a defined monitoring area boundary (convex hull) that encompasses all stations.
These boundaries are interactive and can be toggled on the main map.

## INTERACTIVE FEATURES

### Main Comprehensive Map
- Features:
  * City boundaries with pollution overlay (convex hulls)
  * Heatmap visualization of PM2.5 concentration
  * Interactive marker clustering (zoom in to see stations)
  * Layer control to toggle cities on/off
  * Fullscreen mode
  * Mini map for navigation
  * Color-coded markers by AQI status

### Detailed City Maps
- Zoomed views for each city
- Radius analysis (0.01-0.05 degree buffers around city center)
- Wind direction indicators
- Comprehensive station information on hover
- Heatmap overlays with detailed colors

### Voronoi Diagram
- Shows monitoring coverage areas
- Each station's "service area" (Voronoi region)
- Helps visualize spatial coverage gaps
- Useful for network optimization

## SPATIAL DENSITY ANALYSIS

### Kernel Density Estimation (KDE)
- Identifies pollution hotspots and gradients
- Weights points by PM2.5 concentration
- Shows probability density of pollution across space
- Reveals spatial patterns not visible from individual stations

Results:
- High-density areas = persistent pollution sources
- Low-density areas = well-mixed or clean zones
- Gradients show transport pathways

## CLUSTERING ANALYSIS

Spatial Statistics Summary:
"""

for _, row in clustering_stats_df.iterrows():
    spatial_report += f"""
### {row['City']}
- Number of Stations: {int(row['Num Stations'])}
- Avg Distance to Nearest Neighbor: {row['Avg Dist to Nearest (km)']:.2f} km
- Spatial Dispersion Index: {row['Spatial Dispersion']:.4f}
- Pollution Variability (CV): {row['Pollution CV']:.3f}
- Mean PM2.5: {row['Mean PM2.5']:.2f} µg/m³
- PM2.5 Standard Deviation: {row['PM2.5 Std']:.2f} µg/m³

Interpretation:
- Smaller avg distance → More densely packed stations
- Higher CV → More variable pollution across city
"""

spatial_report += f"""

## PROXIMITY RELATIONSHIPS

Each station's nearest neighbor analysis reveals:
- How well distributed the monitoring network is
- Whether nearby stations record similar pollution levels
- Potential for data redundancy or coverage gaps

Distance Distribution:
"""

for city in ['Ancona', 'Athens', 'Zaragoza']:
    city_prox = proximity_df[proximity_df['City'] == city]
    spatial_report += f"""
{city}:
  - Mean distance to nearest: {city_prox['Distance (km)'].mean():.2f} km
  - Min distance: {city_prox['Distance (km)'].min():.2f} km
  - Max distance: {city_prox['Distance (km)'].max():.2f} km
  - Mean pollution difference: {city_prox['Pollution Diff (PM2.5)'].mean():.2f} µg/m³
"""

spatial_report += f"""

## SPATIAL INSIGHTS & IMPLICATIONS

1. **Network Coverage**
   - Voronoi diagrams show coverage areas
   - Identify areas far from any station
   - Potential locations for new monitors

2. **Pollution Hotspots**
   - KDE reveals consistent high-pollution areas
   - May indicate traffic nodes, industrial zones
   - Target areas for intervention policies

3. **Spatial Correlation**
   - Nearby stations tend to have similar pollution
   - Suggests local sources dominate
   - Important for spatial modeling

4. **Network Efficiency**
   - Average nearest-neighbor distances quantify density
   - Helps optimize monitoring network
   - Data quality consistency across network

## FILES GENERATED

### Interactive Maps
1. 01_interactive_city_boundaries_map.html
   - Main comprehensive map with all cities
   - Layer control to toggle each city
   - Heatmaps + station markers
   - Toggle visibility by city

2. 02_detailed_map_[city].html (x3)
   - Zoomed detail maps for each city
   - Radius analysis rings
   - Wind direction indicators
   - Comprehensive station popups

3. 04_voronoi_diagram_[city].html (x3)
   - Voronoi service areas
   - Shows coverage zones
   - Useful for network planning

### Analysis Visualizations
4. 03_kernel_density_estimation.html
   - PM2.5 spatial density heatmaps
   - Identifies pollution hotspots
   - Contour visualization

5. 05_proximity_analysis.html
   - Distance statistics
   - Pollution difference analysis
   - Box plots by city

### Data Files
6. proximity_analysis.csv - Nearest neighbor analysis
7. clustering_statistics.csv - Spatial clustering metrics

## RECOMMENDATIONS

1. **Use city boundaries for policy enforcement**
   - Define air quality zones within boundaries
   - Implement targeted measures in high-pollution areas

2. **Optimize monitoring network**
   - Use Voronoi to identify coverage gaps
   - Consider filling gaps identified in service areas

3. **Monitor hotspots intensively**
   - KDE shows where pollution is concentrated
   - Deploy mobile monitors or increase frequency there

4. **Coordinate across cities**
   - Understand boundary effects
   - Plan regional air quality strategy

5. **Track network performance**
   - Monitor average nearest-neighbor distance
   - Ensure consistent spatial coverage
"""

In [39]:
# Save report
with open(f'{RESULTS_GEOSPATIAL}/ADVANCED_GEOSPATIAL_REPORT.txt', 'w') as f:
    f.write(spatial_report)

print("✅ Saved: ADVANCED_GEOSPATIAL_REPORT.txt")
print(spatial_report)

✅ Saved: ADVANCED_GEOSPATIAL_REPORT.txt

# ADVANCED GEOSPATIAL & SPATIAL ANALYSIS REPORT
Generated: 2026-07-03 14:52:22

## GEOGRAPHIC OVERVIEW

### City Boundaries & Coverage
- Ancona: 22 monitoring stations
- Athens: 64 monitoring stations
- Zaragoza: 10 monitoring stations

Total: 96 stations across Europe

### Spatial Extent (Convex Hulls)
Each city has a defined monitoring area boundary (convex hull) that encompasses all stations.
These boundaries are interactive and can be toggled on the main map.

## INTERACTIVE FEATURES

### Main Comprehensive Map
- Features:
  * City boundaries with pollution overlay (convex hulls)
  * Heatmap visualization of PM2.5 concentration
  * Interactive marker clustering (zoom in to see stations)
  * Layer control to toggle cities on/off
  * Fullscreen mode
  * Mini map for navigation
  * Color-coded markers by AQI status

### Detailed City Maps
- Zoomed views for each city
- Radius analysis (0.01-0.05 degree buffers around city center)
- Wind directi

# FINAL SUMMARY

In [40]:
print("✅ NOTEBOOK 04 (ADVANCED GEOSPATIAL ANALYSIS) COMPLETE!")

✅ NOTEBOOK 04 (ADVANCED GEOSPATIAL ANALYSIS) COMPLETE!


In [41]:
print(f"""
🗺️  ADVANCED GEOSPATIAL ANALYSIS COMPLETED:
   ✓ Interactive city boundary maps (with layer control)
   ✓ Detailed zoomed maps per city
   ✓ Voronoi diagram service areas (x3)
   ✓ Kernel density estimation analysis
   ✓ Proximity/nearest neighbor analysis
   ✓ Advanced clustering statistics
   ✓ Wind direction visualization
   ✓ Radius/buffer analysis

💾 FILES SAVED ({RESULTS_GEOSPATIAL}/):
   ✓ 01_interactive_city_boundaries_map.html (MAIN - Toggle Cities!)
   ✓ 02_detailed_map_ancona.html
   ✓ 02_detailed_map_athens.html
   ✓ 02_detailed_map_zaragoza.html
   ✓ 03_kernel_density_estimation.html
   ✓ 04_voronoi_diagram_ancona.html
   ✓ 04_voronoi_diagram_athens.html
   ✓ 04_voronoi_diagram_zaragoza.html
   ✓ 05_proximity_analysis.html
   ✓ proximity_analysis.csv
   ✓ clustering_statistics.csv
   ✓ ADVANCED_GEOSPATIAL_REPORT.txt

🎨 ADVANCED FEATURES:
   ✓ City boundary toggles (click legend to show/hide)
   ✓ Multi-layer interactive maps
   ✓ Wind direction indicators
   ✓ Radius/buffer analysis rings
   ✓ Heatmaps dengan progressive colors
   ✓ Voronoi service area visualization
   ✓ Professional legends & controls
   ✓ Fullscreen & minimap support

📊 SPATIAL STATISTICS:
   ✓ Nearest neighbor analysis
   ✓ Clustering metrics
   ✓ Spatial dispersion indices
   ✓ Proximity correlations
   ✓ Network efficiency analysis

🚀 NEXT PHASE:
   → Notebook 05: Pattern Analysis (Meteorological + Temporal Patterns)
   → Notebook 06: Anomaly Detection
   → Notebook 07: Feature Engineering
   → Notebook 08: ML Forecasting

""")

print("✅ Ready for Pattern Analysis!")


🗺️  ADVANCED GEOSPATIAL ANALYSIS COMPLETED:
   ✓ Interactive city boundary maps (with layer control)
   ✓ Detailed zoomed maps per city
   ✓ Voronoi diagram service areas (x3)
   ✓ Kernel density estimation analysis
   ✓ Proximity/nearest neighbor analysis
   ✓ Advanced clustering statistics
   ✓ Wind direction visualization
   ✓ Radius/buffer analysis

💾 FILES SAVED (/content/drive/MyDrive/Project Mandiri/Air Quality Monitoring in European Cities/results/geospatial/):
   ✓ 01_interactive_city_boundaries_map.html (MAIN - Toggle Cities!)
   ✓ 02_detailed_map_ancona.html
   ✓ 02_detailed_map_athens.html
   ✓ 02_detailed_map_zaragoza.html
   ✓ 03_kernel_density_estimation.html
   ✓ 04_voronoi_diagram_ancona.html
   ✓ 04_voronoi_diagram_athens.html
   ✓ 04_voronoi_diagram_zaragoza.html
   ✓ 05_proximity_analysis.html
   ✓ proximity_analysis.csv
   ✓ clustering_statistics.csv
   ✓ ADVANCED_GEOSPATIAL_REPORT.txt

🎨 ADVANCED FEATURES:
   ✓ City boundary toggles (click legend to show/hide)
  